# 01 - Data Audit: Chip Companies Financials

This notebook inspects the raw data without modifying it. Diagnostic calculations use a copy, and nothing is written to data/processed.

In [1]:
from pathlib import Path
import pandas as pd

## Load and preview

In [2]:
cwd = Path.cwd().resolve()
project_root = cwd if (cwd / "data" / "raw").is_dir() else cwd.parent
file_path = project_root / "data" / "raw" / "chip_companies_financials.csv"
if not file_path.is_file():
    raise FileNotFoundError(f"Raw dataset not found: {file_path}")
df_raw = pd.read_csv(file_path)
print(f"File path: {file_path}")
print(f"DataFrame shape: {df_raw.shape}")
display(df_raw.head(5))
display(df_raw.tail(5))
display(df_raw.sample(n=min(5, len(df_raw)), random_state=42))

File path: C:\Users\ALBERT\Desktop\Year 1 sem2\DADS5001_Aj.Thitirat_SUN\Mid-Term Project\semiconductor company datasets\data\raw\chip_companies_financials.csv
DataFrame shape: (617, 10)


,year,company_name,ticker,country_iso3,segment,revenue_usd_bn,operating_margin_pct,operating_income_usd_bn,rd_spend_usd_bn,capex_usd_bn
0,2010,TSMC,TSM,TWN,foundry,13.42,36.9,4.95,1.07,6.04
1,2011,TSMC,TSM,TWN,foundry,16.32,42.8,6.99,1.31,7.35
2,2012,TSMC,TSM,TWN,foundry,17.56,36.1,6.34,1.40,7.90
3,2013,TSMC,TSM,TWN,foundry,21.36,39.1,8.34,1.71,9.61
4,2014,TSMC,TSM,TWN,foundry,23.93,37.4,8.96,1.91,10.77


,year,company_name,ticker,country_iso3,segment,revenue_usd_bn,operating_margin_pct,operating_income_usd_bn,rd_spend_usd_bn,capex_usd_bn
612,2022,Tenstorrent,TNS,USA,fabless_ai,0.05,11.6,0.01,0.02,0.00
613,2023,Tenstorrent,TNS,USA,fabless_ai,0.10,8.3,0.01,0.05,0.00
614,2024,Tenstorrent,TNS,USA,fabless_ai,0.31,11.7,0.04,0.15,0.00
615,2025,Tenstorrent,TNS,USA,fabless_ai,0.74,6.6,0.05,0.37,0.01
616,2026,Tenstorrent,TNS,USA,fabless_ai,1.16,10.3,0.12,0.58,0.01


,year,company_name,ticker,country_iso3,segment,revenue_usd_bn,operating_margin_pct,operating_income_usd_bn,rd_spend_usd_bn,capex_usd_bn
49,2025,GlobalFoundries,GFS,USA,foundry,7.24,37.3,2.70,0.58,3.26
581,2020,Cerebras Systems,CRBR,USA,fabless_ai,0.00,5.5,0.00,0.00,0.00
82,2024,SMIC,981,CHN,foundry,8.20,37.3,3.06,0.66,3.69
304,2025,Infineon,IFX,DEU,idm_automotive,15.69,21.6,3.39,1.88,2.35
109,2017,AMD,AMD,USA,fabless_cpu_gpu,5.64,17.9,1.01,1.13,0.56


## Structure

In [3]:
df_raw.info()
display(pd.Series(df_raw.columns, name="column_name").to_frame())
display(df_raw.dtypes.rename("dtype").to_frame())
display(df_raw.memory_usage(deep=True).rename("memory_bytes").to_frame())
display(df_raw.nunique(dropna=False).rename("unique_values_including_missing").to_frame())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 617 entries, 0 to 616
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   year                     617 non-null    int64  
 1   company_name             617 non-null    object 
 2   ticker                   617 non-null    object 
 3   country_iso3             617 non-null    object 
 4   segment                  617 non-null    object 
 5   revenue_usd_bn           617 non-null    float64
 6   operating_margin_pct     617 non-null    float64
 7   operating_income_usd_bn  617 non-null    float64
 8   rd_spend_usd_bn          617 non-null    float64
 9   capex_usd_bn             617 non-null    float64
dtypes: float64(5), int64(1), object(4)
memory usage: 48.3+ KB


,column_name
0,year
1,company_name
2,ticker
3,country_iso3
4,segment
5,revenue_usd_bn
6,operating_margin_pct
7,operating_income_usd_bn
8,rd_spend_usd_bn
9,capex_usd_bn


,dtype
year,int64
company_name,object
ticker,object
country_iso3,object
segment,object
revenue_usd_bn,float64
operating_margin_pct,float64
operating_income_usd_bn,float64
rd_spend_usd_bn,float64
capex_usd_bn,float64


,memory_bytes
Index,132
year,4936
company_name,36398
ticker,32476
country_iso3,32084
segment,37882
revenue_usd_bn,4936
operating_margin_pct,4936
operating_income_usd_bn,4936
rd_spend_usd_bn,4936


,unique_values_including_missing
year,17
company_name,40
ticker,39
country_iso3,9
segment,20
revenue_usd_bn,537
operating_margin_pct,318
operating_income_usd_bn,407
rd_spend_usd_bn,298
capex_usd_bn,339


## Missing data and duplicates

In [4]:
missing_count = df_raw.isna().sum()
missing_pct = df_raw.isna().mean().mul(100)
display(pd.DataFrame({"missing_count": missing_count, "missing_pct": missing_pct}))
text_columns = df_raw.select_dtypes(include=["object", "string"]).columns
blank_masks = pd.DataFrame({c: df_raw[c].notna() & df_raw[c].astype(str).str.strip().eq("") for c in text_columns})
blank_counts = blank_masks.sum().rename("empty_or_whitespace_only_count")
display(blank_counts.to_frame())
complete_duplicate_mask = df_raw.duplicated(keep=False)
key_duplicate_mask = df_raw.duplicated(subset=["year", "company_name"], keep=False)
print(f"Rows in complete duplicate groups: {complete_duplicate_mask.sum()}")
print(f"Rows in duplicate (year, company_name) groups: {key_duplicate_mask.sum()}")
if complete_duplicate_mask.any():
    display(df_raw.loc[complete_duplicate_mask].sort_values(["year", "company_name"]))
if key_duplicate_mask.any():
    display(df_raw.loc[key_duplicate_mask].sort_values(["year", "company_name"]))

,missing_count,missing_pct
year,0,0.0
company_name,0,0.0
ticker,0,0.0
country_iso3,0,0.0
segment,0,0.0
revenue_usd_bn,0,0.0
operating_margin_pct,0,0.0
operating_income_usd_bn,0,0.0
rd_spend_usd_bn,0,0.0
capex_usd_bn,0,0.0


,empty_or_whitespace_only_count
company_name,0
ticker,0
country_iso3,0
segment,0


Rows in complete duplicate groups: 0
Rows in duplicate (year, company_name) groups: 0


## Numeric inspection

Possible extreme values use the 1.5 × IQR rule. These statistical flags are not claims that values are errors.

In [5]:
numeric_columns = df_raw.select_dtypes(include="number").columns
display(df_raw[numeric_columns].describe().T)
display(pd.DataFrame({
    "minimum": df_raw[numeric_columns].min(),
    "maximum": df_raw[numeric_columns].max(),
    "zero_count": df_raw[numeric_columns].eq(0).sum(),
    "negative_count": df_raw[numeric_columns].lt(0).sum(),
}))
q1 = df_raw[numeric_columns].quantile(0.25)
q3 = df_raw[numeric_columns].quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr
extreme_masks = df_raw[numeric_columns].lt(lower_fence) | df_raw[numeric_columns].gt(upper_fence)
display(pd.DataFrame({"lower_fence": lower_fence, "upper_fence": upper_fence, "possible_extreme_count": extreme_masks.sum()}))
possible_extreme_rows = df_raw.loc[extreme_masks.any(axis=1)].copy()
display(possible_extreme_rows)

,count,mean,std,min,25%,50%,75%,max
year,617.0,2018.401945,4.900026,2010.0,2014.00,2019.00,2023.00,2026.00
revenue_usd_bn,617.0,14.354392,21.474730,0.0,3.43,7.15,15.69,250.68
operating_margin_pct,617.0,30.335818,11.517675,3.3,21.20,30.80,39.60,64.20
operating_income_usd_bn,617.0,4.505154,9.884108,0.0,1.08,2.08,4.90,155.25
rd_spend_usd_bn,617.0,1.963225,4.101473,0.0,0.44,0.97,1.89,62.67
capex_usd_bn,617.0,3.117180,6.031188,0.0,0.39,1.03,2.38,50.96


,minimum,maximum,zero_count,negative_count
year,2010.0,2026.00,0,0
revenue_usd_bn,0.0,250.68,8,0
operating_margin_pct,3.3,64.20,0,0
operating_income_usd_bn,0.0,155.25,14,0
rd_spend_usd_bn,0.0,62.67,8,0
capex_usd_bn,0.0,50.96,28,0


,lower_fence,upper_fence,possible_extreme_count
year,2000.500,2036.500,0
revenue_usd_bn,-14.960,34.080,62
operating_margin_pct,-6.400,67.200,0
operating_income_usd_bn,-4.650,10.630,54
rd_spend_usd_bn,-1.735,4.065,62
capex_usd_bn,-2.595,5.365,87


,year,company_name,ticker,country_iso3,segment,revenue_usd_bn,operating_margin_pct,operating_income_usd_bn,rd_spend_usd_bn,capex_usd_bn
0,2010,TSMC,TSM,TWN,foundry,13.42,36.9,4.95,1.07,6.04
1,2011,TSMC,TSM,TWN,foundry,16.32,42.8,6.99,1.31,7.35
2,2012,TSMC,TSM,TWN,foundry,17.56,36.1,6.34,1.40,7.90
3,2013,TSMC,TSM,TWN,foundry,21.36,39.1,8.34,1.71,9.61
4,2014,TSMC,TSM,TWN,foundry,23.93,37.4,8.96,1.91,10.77
...,...,...,...,...,...,...,...,...,...,...
404,2023,ASML,ASML,NLD,equipment_litho,30.31,35.7,10.81,4.85,1.82
405,2024,ASML,ASML,NLD,equipment_litho,31.81,33.3,10.58,5.09,1.91
406,2025,ASML,ASML,NLD,equipment_litho,38.79,33.5,12.99,6.21,2.33
407,2026,ASML,ASML,NLD,equipment_litho,44.16,35.9,15.87,7.07,2.65


## Categorical inspection

In [6]:
print(f"Unique company count: {df_raw['company_name'].nunique()}")
print(f"Unique ticker count: {df_raw['ticker'].nunique()}")
display(df_raw["country_iso3"].value_counts(dropna=False).rename("row_count").to_frame())
display(df_raw["segment"].value_counts(dropna=False).rename("row_count").to_frame())
display(df_raw.groupby("segment", dropna=False)["company_name"].nunique().rename("unique_company_count").to_frame())

Unique company count: 40
Unique ticker count: 39


,row_count
country_iso3,
USA,318
CHN,70
TWN,51
KOR,51
JPN,51
NLD,34
DEU,17
EUR,17
GBR,8


,row_count
segment,
foundry,102
idm_memory,70
idm_automotive,51
fabless_ai,37
fabless_diversified,34
fabless_mobile,34
idm_analog,34
equipment_diversified,34
eda_software,34


,unique_company_count
segment,
eda_software,2
equipment_cleaning,1
equipment_diversified,2
equipment_etch,1
equipment_inspection,1
equipment_litho,1
fabless_ai,5
fabless_cpu_gpu,1
fabless_diversified,2


## Time coverage

In [7]:
print(f"Minimum year: {df_raw['year'].min()}")
print(f"Maximum year: {df_raw['year'].max()}")
display(df_raw["year"].value_counts().sort_index().rename("row_count").to_frame())
company_time_coverage = df_raw.groupby("company_name", dropna=False)["year"].agg(minimum_year="min", maximum_year="max", row_count="size", unique_year_count="nunique")
display(company_time_coverage)
df = df_raw.copy()
coverage_records = []
for company, group in df.groupby("company_name", dropna=False):
    observed = set(group["year"].dropna().astype(int))
    expected = set(range(min(observed), max(observed) + 1)) if observed else set()
    missing_years = sorted(expected - observed)
    if missing_years:
        coverage_records.append({"company_name": company, "missing_years": missing_years, "missing_year_count": len(missing_years)})
companies_with_year_gaps = pd.DataFrame(coverage_records, columns=["company_name", "missing_years", "missing_year_count"])
display(companies_with_year_gaps)

Minimum year: 2010
Maximum year: 2026


,row_count
year,
2010,33
2011,33
2012,33
2013,33
2014,33
2015,33
2016,33
2017,34
2018,35


,minimum_year,maximum_year,row_count,unique_year_count
company_name,,,,
AMD,2010,2026,17,17
ASML,2010,2026,17,17
Analog Devices,2010,2026,17,17
Applied Materials,2010,2026,17,17
Broadcom,2010,2026,17,17
Cadence,2010,2026,17,17
Cerebras Systems,2019,2026,8,8
ChangXin Memory (CXMT),2018,2026,9,9
GlobalFoundries,2010,2026,17,17


,company_name,missing_years,missing_year_count


## Consistency checks

Checks identify observations for review without correcting the data.

In [8]:
year_out_of_range_mask = ~df["year"].between(2010, 2026, inclusive="both")
invalid_country_mask = ~df["country_iso3"].fillna("").astype(str).str.fullmatch(r"[A-Za-z]{3}")
eur_mask = df["country_iso3"].eq("EUR")
print(f"Rows with year outside 2010–2026: {year_out_of_range_mask.sum()}")
display(df.loc[year_out_of_range_mask])
print(f"Rows with country_iso3 not exactly three alphabetic characters: {invalid_country_mask.sum()}")
display(df.loc[invalid_country_mask, ["year", "company_name", "country_iso3"]])
print(f"Rows separately flagged as EUR: {eur_mask.sum()}")
display(df.loc[eur_mask, ["year", "company_name", "country_iso3"]])

Rows with year outside 2010–2026: 0


,year,company_name,ticker,country_iso3,segment,revenue_usd_bn,operating_margin_pct,operating_income_usd_bn,rd_spend_usd_bn,capex_usd_bn


Rows with country_iso3 not exactly three alphabetic characters: 0


,year,company_name,country_iso3


Rows separately flagged as EUR: 17


,year,company_name,country_iso3
306,2010,STMicroelectronics,EUR
307,2011,STMicroelectronics,EUR
308,2012,STMicroelectronics,EUR
309,2013,STMicroelectronics,EUR
310,2014,STMicroelectronics,EUR
311,2015,STMicroelectronics,EUR
312,2016,STMicroelectronics,EUR
313,2017,STMicroelectronics,EUR
314,2018,STMicroelectronics,EUR
315,2019,STMicroelectronics,EUR


In [9]:
df["calculated_operating_income_usd_bn"] = df["revenue_usd_bn"] * df["operating_margin_pct"] / 100
df["operating_income_abs_difference_usd_bn"] = (df["operating_income_usd_bn"] - df["calculated_operating_income_usd_bn"]).abs()
operating_income_difference_mask = df["operating_income_abs_difference_usd_bn"].gt(0.02)
print(f"Rows with operating-income absolute difference > 0.02 billion USD: {operating_income_difference_mask.sum()}")
display(df.loc[operating_income_difference_mask, ["year", "company_name", "revenue_usd_bn", "operating_margin_pct", "operating_income_usd_bn", "calculated_operating_income_usd_bn", "operating_income_abs_difference_usd_bn"]])
display(df["operating_income_abs_difference_usd_bn"].describe().to_frame())

Rows with operating-income absolute difference > 0.02 billion USD: 18


,year,company_name,revenue_usd_bn,operating_margin_pct,operating_income_usd_bn,calculated_operating_income_usd_bn,operating_income_abs_difference_usd_bn
9,2019,TSMC,40.91,39.9,16.30,16.32309,0.02309
11,2021,TSMC,62.97,39.5,24.90,24.87315,0.02685
12,2022,TSMC,74.93,38.9,29.18,29.14777,0.03223
15,2025,TSMC,108.13,38.8,41.93,41.95444,0.02444
16,2026,TSMC,113.25,37.6,42.53,42.58200,0.05200
100,2025,NVIDIA,192.19,64.2,123.43,123.38598,0.04402
101,2026,NVIDIA,250.68,61.9,155.25,155.17092,0.07908
152,2026,Broadcom,69.87,26.8,18.75,18.72516,0.02484
191,2014,Intel,50.47,16.2,8.20,8.17614,0.02386
194,2017,Intel,66.18,15.8,10.43,10.45644,0.02644


,operating_income_abs_difference_usd_bn
count,617.000000
mean,0.004659
std,0.006525
min,0.000000
25%,0.001400
50%,0.003080
75%,0.005210
max,0.079080


In [10]:
nonzero_revenue = df["revenue_usd_bn"].where(df["revenue_usd_bn"].ne(0))
df["rd_intensity_pct"] = df["rd_spend_usd_bn"].div(nonzero_revenue).mul(100)
df["capex_intensity_pct"] = df["capex_usd_bn"].div(nonzero_revenue).mul(100)
print(f"Rows with zero revenue (ratios left missing): {df['revenue_usd_bn'].eq(0).sum()}")
display(df[["rd_intensity_pct", "capex_intensity_pct"]].describe().T)
ratio_constancy = df.groupby("company_name", dropna=False).agg(rd_years=("rd_intensity_pct", "count"), rd_min_pct=("rd_intensity_pct", "min"), rd_max_pct=("rd_intensity_pct", "max"), capex_years=("capex_intensity_pct", "count"), capex_min_pct=("capex_intensity_pct", "min"), capex_max_pct=("capex_intensity_pct", "max"))
ratio_constancy["rd_range_pp"] = ratio_constancy["rd_max_pct"] - ratio_constancy["rd_min_pct"]
ratio_constancy["capex_range_pp"] = ratio_constancy["capex_max_pct"] - ratio_constancy["capex_min_pct"]
ratio_constancy["rd_nearly_constant"] = ratio_constancy["rd_years"].ge(5) & ratio_constancy["rd_range_pp"].le(0.1)
ratio_constancy["capex_nearly_constant"] = ratio_constancy["capex_years"].ge(5) & ratio_constancy["capex_range_pp"].le(0.1)
nearly_constant_ratios = ratio_constancy.loc[ratio_constancy["rd_nearly_constant"] | ratio_constancy["capex_nearly_constant"]].copy()
print("Nearly constant definition: at least 5 ratios and range <= 0.1 percentage points.")
display(nearly_constant_ratios)

Rows with zero revenue (ratios left missing): 8


,count,mean,std,min,25%,50%,75%,max
rd_intensity_pct,609.0,15.143145,10.379782,7.352941,10.008028,11.995104,12.086093,100.000000
capex_intensity_pct,609.0,18.394012,15.068915,0.000000,9.976976,10.071942,34.983790,45.714286


Nearly constant definition: at least 5 ratios and range <= 0.1 percentage points.


,rd_years,rd_min_pct,rd_max_pct,capex_years,capex_min_pct,capex_max_pct,rd_range_pp,capex_range_pp,rd_nearly_constant,capex_nearly_constant
company_name,,,,,,,,,,
AMD,17,19.949495,20.040692,17,9.922179,10.101010,0.091197,0.178831,True,False
Applied Materials,17,11.951754,12.034079,17,9.972299,10.041841,0.082324,0.069542,True,True
Graphcore,6,40.000000,57.142857,6,0.000000,0.000000,17.142857,0.000000,False,True
Intel,17,19.992074,20.009023,17,24.989049,25.009870,0.016949,0.020820,True,True
Micron,17,9.960159,10.034305,17,34.954407,35.017422,0.074146,0.063014,True,True
Qualcomm,17,11.979167,12.018779,17,9.968254,10.046948,0.039613,0.078694,True,True
Renesas,17,11.933535,12.040558,17,14.934114,15.033948,0.107023,0.099833,False,True
SK Hynix,17,9.973753,10.017271,17,34.973753,35.061263,0.043518,0.087510,True,True
STMicroelectronics,17,11.955420,12.050985,17,9.965238,10.056657,0.095564,0.091420,True,True


## Data Audit Summary

In [11]:
summary_rows = [
    ["Missing values", "FLAG" if missing_count.sum() else "PASS", int(df_raw.isna().any(axis=1).sum()), "Rows with at least one missing value.", "Review meaning and source before treatment."],
    ["Blank text values", "FLAG" if blank_counts.sum() else "PASS", int(blank_masks.any(axis=1).sum()), "Rows with empty or whitespace-only text.", "Confirm whether blanks represent missing data."],
    ["Complete duplicate rows", "FLAG" if complete_duplicate_mask.any() else "PASS", int(complete_duplicate_mask.sum()), "Rows in exact duplicate groups.", "Verify provenance before considering removal."],
    ["Duplicate year-company keys", "FLAG" if key_duplicate_mask.any() else "PASS", int(key_duplicate_mask.sum()), "Rows sharing the proposed key.", "Investigate repeated keys before preprocessing."],
    ["Year range 2010–2026", "FLAG" if year_out_of_range_mask.any() else "PASS", int(year_out_of_range_mask.sum()), "Rows outside the project period.", "Verify against the source."],
    ["Country-code format", "FLAG" if invalid_country_mask.any() else "PASS", int(invalid_country_mask.sum()), "Rows not matching three alphabetic characters.", "Compare with an authoritative code reference."],
    ["EUR country-code flag", "FLAG" if eur_mask.any() else "PASS", int(eur_mask.sum()), "EUR is a currency code, not a country code.", "Investigate intended geography."],
    ["Operating-income tolerance", "FLAG" if operating_income_difference_mask.any() else "PASS", int(operating_income_difference_mask.sum()), "Rows with absolute difference above 0.02 billion USD.", "Review rounding, definitions, and sources."],
    ["Internal company-year gaps", "FLAG" if len(companies_with_year_gaps) else "PASS", int(len(companies_with_year_gaps)), "Companies with gaps inside their own year range.", "Account for panel gaps in comparisons."],
    ["Possible numeric extremes", "REVIEW" if len(possible_extreme_rows) else "PASS", int(len(possible_extreme_rows)), "1.5 × IQR statistical flags, not proven errors.", "Review in business context and against sources."],
    ["Nearly constant R&D or CapEx ratios", "REVIEW" if len(nearly_constant_ratios) else "PASS", int(len(nearly_constant_ratios)), "Companies meeting the documented threshold.", "Investigate reported, estimated, or generated patterns."],
]
audit_summary = pd.DataFrame(summary_rows, columns=["check_name", "result", "affected_rows", "interpretation", "recommended_action"])
display(audit_summary)

,check_name,result,affected_rows,interpretation,recommended_action
0,Missing values,PASS,0,Rows with at least one missing value.,Review meaning and source before treatment.
1,Blank text values,PASS,0,Rows with empty or whitespace-only text.,Confirm whether blanks represent missing data.
2,Complete duplicate rows,PASS,0,Rows in exact duplicate groups.,Verify provenance before considering removal.
3,Duplicate year-company keys,PASS,0,Rows sharing the proposed key.,Investigate repeated keys before preprocessing.
4,Year range 2010–2026,PASS,0,Rows outside the project period.,Verify against the source.
5,Country-code format,PASS,0,Rows not matching three alphabetic characters.,Compare with an authoritative code reference.
6,EUR country-code flag,FLAG,17,"EUR is a currency code, not a country code.",Investigate intended geography.
7,Operating-income tolerance,FLAG,18,Rows with absolute difference above 0.02 billi...,"Review rounding, definitions, and sources."
8,Internal company-year gaps,PASS,0,Companies with gaps inside their own year range.,Account for panel gaps in comparisons.
9,Possible numeric extremes,REVIEW,110,"1.5 × IQR statistical flags, not proven errors.",Review in business context and against sources.


## Audit Conclusions

**Confirmed findings:** The dataset contains 617 rows and 10 columns. The audit found no missing values, blank text values, complete duplicate rows, duplicate year-company keys, years outside 2010–2026, malformed three-alphabetic-character codes, or internal gaps within each company’s observed year range. It confirmed 17 rows coded as EUR, which is a currency code rather than a country code, and 18 rows where reported operating income differs from revenue × operating margin / 100 by more than 0.02 billion USD.

**Suspected issues requiring validation:** The 1.5 × IQR diagnostic flagged 110 rows with possible numeric extremes, and 12 companies met the stated nearly-constant R&D or CapEx ratio threshold (at least five observations with a range no greater than 0.1 percentage points). These are review signals, not evidence that values are erroneous. The EUR values and operating-income differences also require checking against source definitions before any correction. No raw values were changed.